# Camada Gold — Indicadores Analíticos

Este notebook implementa a camada Gold da pipeline de dados de alfabetização.

A Gold utiliza os dados tratados e validados da camada Silver para produzir tabelas analíticas voltadas ao acompanhamento do desempenho educacional, comparação com metas de alfabetização e apoio à tomada de decisão.

As tabelas resultantes poderão ser utilizadas em análises, dashboards, políticas públicas e aplicações futuras de Inteligência Artificial.

## 1. Perguntas analíticas

A camada Gold foi estruturada para responder às seguintes perguntas:

1. Qual é a taxa de alfabetização por UF e município?
2. Como os indicadores evoluíram entre os anos disponíveis?
3. Quais localidades apresentam os melhores e piores resultados?
4. Quais localidades estão abaixo ou acima das metas estabelecidas?
5. Qual é a distância entre o resultado observado e a meta de alfabetização?
6. Qual é a participação dos alunos nas avaliações?
7. Qual é a proporção de alunos alfabetizados entre os alunos com avaliação válida?
8. Quais regiões podem demandar maior atenção para políticas públicas?

## 2. Indicadores da Gold

Serão construídos indicadores como:

- taxa de alfabetização;
- meta de alfabetização;
- diferença entre taxa observada e meta;
- status de atingimento da meta;
- evolução da taxa de alfabetização;
- percentual de participação;
- quantidade de alunos avaliados;
- quantidade e percentual de alunos alfabetizados;
- média de proficiência dos alunos com avaliação válida.

In [0]:
# Define os caminhos das camadas Silver e Gold no Amazon S3.

BUCKET_NAME = "literacy-data-pipeline-tcf2-480749290106-us-east-1-an"

SILVER_PATH = f"s3://{BUCKET_NAME}/silver/"
GOLD_PATH = f"s3://{BUCKET_NAME}/gold/"

print(f"Silver: {SILVER_PATH}")
print(f"Gold: {GOLD_PATH}")

In [0]:
# Importa as funções do PySpark utilizadas nas transformações e agregações da Gold.

from pyspark.sql import functions as F

In [0]:
# Mapeia os datasets Parquet da Silver utilizados na construção da Gold.

DATASETS_SILVER = {
    "avaliacao_alfabetizacao_municipio":
        f"{SILVER_PATH}avaliacao_alfabetizacao_municipio/",

    "avaliacao_alfabetizacao_uf":
        f"{SILVER_PATH}avaliacao_alfabetizacao_uf/",

    "avaliacao_alunos":
        f"{SILVER_PATH}avaliacao_alunos/",

    "meta_alfabetizacao_brasil":
        f"{SILVER_PATH}meta_alfabetizacao_brasil/",

    "meta_alfabetizacao_municipio":
        f"{SILVER_PATH}meta_alfabetizacao_municipio/",

    "meta_alfabetizacao_uf":
        f"{SILVER_PATH}meta_alfabetizacao_uf/"
}

In [0]:
# Carrega os datasets tratados da Silver em formato Parquet.

dataframes_silver = {}

for nome, caminho in DATASETS_SILVER.items():
    try:
        df = spark.read.parquet(caminho)

        dataframes_silver[nome] = df

        print(
            f"[OK] {nome}: "
            f"{df.count()} registros | "
            f"{len(df.columns)} colunas"
        )

    except Exception as erro:
        print(f"[ERRO] {nome}: {erro}")

## 3. Gold — Indicadores por UF

Nesta etapa é construída a visão analítica por Unidade Federativa.

Antes da integração entre avaliação e metas, é validada a granularidade da base de avaliação para garantir que a comparação seja realizada entre registros semanticamente equivalentes e sem multiplicação indevida de dados.

In [0]:
# Define os DataFrames utilizados na construção dos indicadores por UF.

df_avaliacao_uf = dataframes_silver["avaliacao_alfabetizacao_uf"]
df_meta_uf = dataframes_silver["meta_alfabetizacao_uf"]

print(
    f"[OK] Bases UF carregadas: "
    f"{df_avaliacao_uf.count()} registros de avaliação | "
    f"{df_meta_uf.count()} registros de metas"
)

In [0]:
# Analisa a distribuição dos códigos de rede por ano na avaliação por UF para identificar qual granularidade deve ser utilizada na Gold estadual.

df_avaliacao_uf = dataframes_silver["avaliacao_alfabetizacao_uf"]

(
    df_avaliacao_uf
    .groupBy("ano", "rede")
    .count()
    .orderBy("ano", "rede")
    .display()
)

In [0]:
# Exibe a quantidade de UFs distintas por ano e código de rede para avaliar a cobertura de cada recorte antes da integração com as metas.

(
    df_avaliacao_uf
    .groupBy("ano", "rede")
    .agg(
        F.countDistinct("sigla_uf").alias("ufs")
    )
    .orderBy("ano", "rede")
    .display()
)

### 3.1 Validação das redes de ensino

A base de avaliação classifica a rede de ensino por códigos, enquanto a base de metas utiliza a categoria consolidada "Pública".

O mapeamento considerado é:

- 2 — Estadual
- 3 — Municipal
- 5 — Privada
- 0 — Não aplicável / Outros

Antes da construção do indicador estadual, é necessário avaliar como as redes públicas Estadual e Municipal devem ser consolidadas para manter equivalência semântica com as metas.

In [0]:
# Analisa as taxas de alfabetização das redes Estadual e Municipal antes da definição da regra de consolidação da rede Pública.

(
    df_avaliacao_uf
    .filter(F.col("rede").isin(2, 3))
    .select(
        "ano",
        "sigla_uf",
        "rede",
        "serie",
        "taxa_alfabetizacao",
        "media_portugues"
    )
    .orderBy("ano", "sigla_uf", "rede")
    .display()
)

In [0]:
# Verifica a quantidade de registros existentes por UF e ano considerando somente as redes públicas Estadual e Municipal.

(
    df_avaliacao_uf
    .filter(F.col("rede").isin(2, 3))
    .groupBy("ano", "sigla_uf")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("rede").alias("redes_publicas")
    )
    .orderBy("ano", "sigla_uf")
    .display()
)

In [0]:
# Tabela que adiciona a descrição da rede de ensino e seleciona os indicadores relevantes para a visão analítica por UF.

MAPA_REDE = (
    F.when(F.col("rede") == 2, "Estadual")
     .when(F.col("rede") == 3, "Municipal")
     .when(F.col("rede") == 5, "Privada")
     .when(F.col("rede") == 0, "Outros")
     .otherwise("Desconhecida")
)

gold_uf_desempenho = (
    df_avaliacao_uf
    .withColumn("rede_ensino", MAPA_REDE)
    .select(
        "ano",
        "sigla_uf",
        "rede",
        "rede_ensino",
        "serie",
        "taxa_alfabetizacao",
        "media_portugues"
    )
)

display(
    gold_uf_desempenho
    .orderBy("ano", "sigla_uf", "rede")
)

In [0]:
# Valida se ano, UF e rede identificam unicamente os registros da tabela analítica estadual.

duplicidades_gold_uf = (
    gold_uf_desempenho
    .groupBy("ano", "sigla_uf", "rede")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Chaves duplicadas: {duplicidades_gold_uf}")

print(
    f"Registros Gold UF: {gold_uf_desempenho.count()}"
)

## 4. Gold — Indicadores por Município

Nesta etapa é construída a visão analítica municipal, relacionando os resultados observados de alfabetização com as metas estabelecidas.

A integração considera exclusivamente a rede Municipal (`rede = 3`) da base de avaliação, garantindo compatibilidade com a granularidade da base de metas municipais.

A partir dessa integração serão derivados indicadores de distância para a meta e situação de atingimento.

In [0]:
# Define os DataFrames utilizados na construção dos indicadores municipais.

df_avaliacao_municipio = dataframes_silver[
    "avaliacao_alfabetizacao_municipio"
]

df_meta_municipio = dataframes_silver[
    "meta_alfabetizacao_municipio"
]

print(
    f"[OK] Bases municipais carregadas: "
    f"{df_avaliacao_municipio.count()} registros de avaliação | "
    f"{df_meta_municipio.count()} registros de metas"
)

In [0]:
# Seleciona somente os resultados da rede Municipal (rede = 3), compatível com a granularidade da base de metas municipais.

avaliacao_municipal = (
    df_avaliacao_municipio
    .filter(F.col("rede") == 3)
    .select(
        "ano",
        "id_municipio",
        "serie",
        "taxa_alfabetizacao",
        "media_portugues"
    )
)

print(
    f"[OK] Avaliação municipal: "
    f"{avaliacao_municipal.count()} registros"
)

In [0]:
# Valida a unicidade de ano + município nas duas bases antes da integração.

for nome, df in {
    "avaliacao_municipal": avaliacao_municipal,
    "meta_municipio": df_meta_municipio
}.items():

    duplicidades = (
        df
        .groupBy("ano", "id_municipio")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"{nome}: {duplicidades} chaves duplicadas")

In [0]:
# Integra os resultados da rede Municipal com as metas de alfabetização utilizando ano e município como chaves da comparação.

gold_municipio_desempenho = (
    avaliacao_municipal.alias("avaliacao")
    .join(
        df_meta_municipio.alias("meta"),
        ["ano", "id_municipio"],
        "left"
    )
    .select(
        "ano",
        "id_municipio",
        F.col("avaliacao.serie").alias("serie"),
        F.col("avaliacao.taxa_alfabetizacao").alias("taxa_alfabetizacao"),
        F.col("avaliacao.media_portugues").alias("media_portugues"),
        F.col("meta.percentual_participacao").alias("percentual_participacao"),
        F.col("meta.nivel_alfabetizacao").alias("nivel_alfabetizacao"),
        F.col("meta.meta_alfabetizacao_2024").alias("meta_alfabetizacao_2024"),
        F.col("meta.meta_alfabetizacao_2025").alias("meta_alfabetizacao_2025"),
        F.col("meta.meta_alfabetizacao_2026").alias("meta_alfabetizacao_2026"),
        F.col("meta.meta_alfabetizacao_2027").alias("meta_alfabetizacao_2027"),
        F.col("meta.meta_alfabetizacao_2028").alias("meta_alfabetizacao_2028"),
        F.col("meta.meta_alfabetizacao_2029").alias("meta_alfabetizacao_2029"),
        F.col("meta.meta_alfabetizacao_2030").alias("meta_alfabetizacao_2030")
    )
)

In [0]:
# Define a meta correspondente ao ano da avaliação.
# Para anos sem meta disponível, o valor permanece nulo.

gold_municipio_desempenho = (
    gold_municipio_desempenho
    .withColumn(
        "meta_ano",
        F.when(
            F.col("ano") == 2024,
            F.col("meta_alfabetizacao_2024")
        )
    )
)

In [0]:
# Calcula a distância entre o resultado observado e a meta do ano e classifica a situação de atingimento da meta.

gold_municipio_desempenho = (
    gold_municipio_desempenho
    .withColumn(
        "diferenca_para_meta",
        F.when(
            F.col("meta_ano").isNotNull(),
            F.round(
                F.col("taxa_alfabetizacao") - F.col("meta_ano"),
                2
            )
        )
    )
    .withColumn(
        "status_meta",
        F.when(
            F.col("meta_ano").isNull(),
            "SEM_META"
        )
        .when(
            F.col("taxa_alfabetizacao") >= F.col("meta_ano"),
            "ATINGIDA"
        )
        .otherwise("NAO_ATINGIDA")
    )
)

In [0]:
# Exibe uma amostra da comparação entre desempenho e meta em 2024.

display(
    gold_municipio_desempenho
    .filter(F.col("ano") == 2024)
    .select(
        "ano",
        "id_municipio",
        "taxa_alfabetizacao",
        "meta_ano",
        "diferenca_para_meta",
        "status_meta"
    )
    .orderBy("diferenca_para_meta")
)

### 4.1 Indicadores consolidados de desempenho municipal

A partir da comparação entre os resultados observados e as metas, são calculados indicadores consolidados para avaliar o desempenho dos municípios em 2024.

Os indicadores permitem identificar a quantidade de municípios com meta disponível, o percentual que atingiu a meta e a distância média entre o resultado observado e o valor esperado.

In [0]:
# Consolida a quantidade de municípios por situação de atingimento da meta.

resumo_status_meta = (
    gold_municipio_desempenho
    .groupBy("ano", "status_meta")
    .agg(
        F.count("*").alias("quantidade_municipios")
    )
    .orderBy("ano", "status_meta")
)

display(resumo_status_meta)

In [0]:
# Calcula indicadores consolidados de desempenho dos municípios em 2024.

gold_kpis_municipais = (
    gold_municipio_desempenho
    .filter(
        (F.col("ano") == 2024)
        & F.col("meta_ano").isNotNull()
    )
    .agg(
        F.count("*").alias("municipios_com_meta"),

        F.sum(
            F.when(F.col("status_meta") == "ATINGIDA", 1)
            .otherwise(0)
        ).alias("municipios_meta_atingida"),

        F.sum(
            F.when(F.col("status_meta") == "NAO_ATINGIDA", 1)
            .otherwise(0)
        ).alias("municipios_meta_nao_atingida"),

        F.round(
            F.avg("taxa_alfabetizacao"), 2
        ).alias("taxa_alfabetizacao_media"),

        F.round(
            F.avg("meta_ano"), 2
        ).alias("meta_media"),

        F.round(
            F.avg("diferenca_para_meta"), 2
        ).alias("diferenca_media_para_meta")
    )
    .withColumn(
        "percentual_municipios_meta_atingida",
        F.round(
            F.col("municipios_meta_atingida")
            / F.col("municipios_com_meta")
            * 100,
            2
        )
    )
)

display(gold_kpis_municipais)

### 4.2 Evolução da alfabetização entre 2023 e 2024

A evolução municipal é calculada comparando a taxa de alfabetização de cada município entre os anos de 2023 e 2024.

O indicador considera apenas municípios com resultados disponíveis nos dois anos, evitando comparar localidades sem histórico equivalente.

In [0]:
from pyspark.sql.window import Window

In [0]:
# Calcula a evolução anual da taxa de alfabetização por município utilizando o resultado do ano anterior como referência.

janela_municipio = (
    Window
    .partitionBy("id_municipio")
    .orderBy("ano")
)

gold_municipio_evolucao = (
    avaliacao_municipal
    .withColumn(
        "taxa_ano_anterior",
        F.lag("taxa_alfabetizacao").over(janela_municipio)
    )
    .withColumn(
        "evolucao_taxa",
        F.when(
            F.col("taxa_ano_anterior").isNotNull(),
            F.round(
                F.col("taxa_alfabetizacao")
                - F.col("taxa_ano_anterior"),
                2
            )
        )
    )
)

In [0]:
# Seleciona os municípios com resultados comparáveis entre 2023 e 2024 e classifica a evolução da taxa de alfabetização.

gold_municipio_evolucao_2024 = (
    gold_municipio_evolucao
    .filter(
        (F.col("ano") == 2024)
        & F.col("taxa_ano_anterior").isNotNull()
    )
    .withColumn(
        "status_evolucao",
        F.when(F.col("evolucao_taxa") > 0, "MELHORA")
         .when(F.col("evolucao_taxa") < 0, "QUEDA")
         .otherwise("ESTAVEL")
    )
)

display(
    gold_municipio_evolucao_2024
    .select(
        "id_municipio",
        "taxa_ano_anterior",
        "taxa_alfabetizacao",
        "evolucao_taxa",
        "status_evolucao"
    )
    .orderBy("evolucao_taxa")
)

In [0]:
# Consolida os indicadores de evolução municipal entre 2023 e 2024.

gold_kpis_evolucao = (
    gold_municipio_evolucao_2024
    .agg(
        F.count("*").alias("municipios_comparaveis"),

        F.sum(
            F.when(F.col("status_evolucao") == "MELHORA", 1)
             .otherwise(0)
        ).alias("municipios_com_melhora"),

        F.sum(
            F.when(F.col("status_evolucao") == "QUEDA", 1)
             .otherwise(0)
        ).alias("municipios_com_queda"),

        F.sum(
            F.when(F.col("status_evolucao") == "ESTAVEL", 1)
             .otherwise(0)
        ).alias("municipios_estaveis"),

        F.round(
            F.avg("evolucao_taxa"), 2
        ).alias("evolucao_media")
    )
    .withColumn(
        "percentual_municipios_com_melhora",
        F.round(
            F.col("municipios_com_melhora")
            / F.col("municipios_comparaveis")
            * 100,
            2
        )
    )
)

display(gold_kpis_evolucao)

## 5. Gold — Indicadores dos Alunos

Nesta etapa são construídos indicadores a partir da base individual de alunos.

As métricas consideram presença, preenchimento do caderno, alfabetização e proficiência, respeitando as regras de qualidade identificadas anteriormente. Alunos sem preenchimento válido do caderno não são utilizados nos cálculos de proficiência.

In [0]:
# Define a base de alunos utilizada na construção dos indicadores analíticos.

df_alunos = dataframes_silver["avaliacao_alunos"]

print(
    f"[OK] Base de alunos carregada: "
    f"{df_alunos.count()} registros"
)

In [0]:
# Consolida indicadores dos alunos por ano e município, considerando presença, participação, alfabetização e proficiência.

gold_indicadores_alunos = (
    df_alunos
    .groupBy("ano", "id_municipio")
    .agg(
        F.count("*").alias("alunos_total"),

        F.sum(
            F.when(F.col("presenca") == 1, 1).otherwise(0)
        ).alias("alunos_presentes"),

        F.sum(
            F.when(F.col("preenchimento_caderno") == 1, 1).otherwise(0)
        ).alias("alunos_avaliacao_valida"),

        F.sum(
            F.when(
                (F.col("preenchimento_caderno") == 1)
                & (F.col("alfabetizado") == 1),
                1
            ).otherwise(0)
        ).alias("alunos_alfabetizados"),

        F.round(
            F.avg(
                F.when(
                    F.col("preenchimento_caderno") == 1,
                    F.col("proficiencia")
                )
            ),
            2
        ).alias("proficiencia_media")
    )
)

In [0]:
# Calcula os percentuais de presença, participação válida e alfabetização entre os alunos avaliados.

gold_indicadores_alunos = (
    gold_indicadores_alunos
    .withColumn(
        "percentual_presenca",
        F.round(
            F.col("alunos_presentes")
            / F.col("alunos_total")
            * 100,
            2
        )
    )
    .withColumn(
        "percentual_avaliacao_valida",
        F.round(
            F.col("alunos_avaliacao_valida")
            / F.col("alunos_total")
            * 100,
            2
        )
    )
    .withColumn(
        "percentual_alfabetizados",
        F.when(
            F.col("alunos_avaliacao_valida") > 0,
            F.round(
                F.col("alunos_alfabetizados")
                / F.col("alunos_avaliacao_valida")
                * 100,
                2
            )
        )
    )
)

In [0]:
# Exibe uma amostra dos indicadores educacionais derivados da base de alunos.

display(
    gold_indicadores_alunos
    .orderBy("ano", "id_municipio")
)

## 6. Gold — Indicadores Nacionais

Nesta etapa é construída a visão nacional dos indicadores de alfabetização.

A base nacional de metas é utilizada diretamente por já disponibilizar os resultados consolidados da rede Pública, permitindo acompanhar a taxa de alfabetização e sua relação com as metas nacionais ao longo dos anos.

In [0]:
# Define a base nacional utilizada na construção dos indicadores da Gold.

df_brasil = dataframes_silver["meta_alfabetizacao_brasil"]

print(
    f"[OK] Base Brasil carregada: "
    f"{df_brasil.count()} registros"
)

display(df_brasil.orderBy("ano"))

In [0]:
# Define a meta correspondente ao ano de cada resultado nacional, preservando as metas disponíveis na própria observação daquele ano.

gold_brasil_desempenho = (
    df_brasil
    .withColumn(
        "meta_ano",
        F.when(
            F.col("ano") == 2024,
            F.col("meta_alfabetizacao_2024")
        )
        .when(
            F.col("ano") == 2025,
            F.col("meta_alfabetizacao_2025")
        )
    )
)

In [0]:
# Calcula a distância entre a taxa nacional observada e a meta correspondente e classifica o resultado quanto ao atingimento da meta.

gold_brasil_desempenho = (
    gold_brasil_desempenho
    .withColumn(
        "diferenca_para_meta",
        F.when(
            F.col("meta_ano").isNotNull(),
            F.round(
                F.col("taxa_alfabetizacao") - F.col("meta_ano"),
                2
            )
        )
    )
    .withColumn(
        "status_meta",
        F.when(
            F.col("meta_ano").isNull(),
            "SEM_META"
        )
        .when(
            F.col("taxa_alfabetizacao") >= F.col("meta_ano"),
            "ATINGIDA"
        )
        .otherwise("NAO_ATINGIDA")
    )
)

In [0]:
# Exibe a evolução nacional e a comparação entre resultado e meta.

display(
    gold_brasil_desempenho
    .select(
        "ano",
        "rede",
        "taxa_alfabetizacao",
        "meta_ano",
        "diferenca_para_meta",
        "status_meta",
        "percentual_participacao"
    )
    .orderBy("ano")
)

## 7. Validação das tabelas Gold

Antes da persistência, as tabelas analíticas são validadas quanto à quantidade de registros, granularidade e unicidade das chaves principais.

Essa etapa garante que as saídas finais da camada Gold estejam consistentes e prontas para consumo analítico.

In [0]:
# Consolida as tabelas finais que serão persistidas na camada Gold.

TABELAS_GOLD = {
    "gold_brasil_desempenho": gold_brasil_desempenho,
    "gold_uf_desempenho": gold_uf_desempenho,
    "gold_municipio_desempenho": gold_municipio_desempenho,
    "gold_municipio_evolucao": gold_municipio_evolucao_2024,
    "gold_indicadores_alunos": gold_indicadores_alunos
}

for nome, df in TABELAS_GOLD.items():
    print(
        f"[OK] {nome}: "
        f"{df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

In [0]:
# Define e valida as chaves das tabelas Gold de acordo com sua granularidade.

CHAVES_GOLD = {
    "gold_brasil_desempenho": ["ano", "rede"],
    "gold_uf_desempenho": ["ano", "sigla_uf", "rede"],
    "gold_municipio_desempenho": ["ano", "id_municipio"],
    "gold_municipio_evolucao": ["ano", "id_municipio"],
    "gold_indicadores_alunos": ["ano", "id_municipio"]
}

resultado_chaves_gold = []

for nome, chaves in CHAVES_GOLD.items():
    df = TABELAS_GOLD[nome]

    duplicadas = (
        df
        .groupBy(*chaves)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    resultado_chaves_gold.append({
        "tabela": nome,
        "chaves_duplicadas": duplicadas
    })

df_validacao_gold = spark.createDataFrame(resultado_chaves_gold)

display(df_validacao_gold.orderBy("tabela"))

## 8. Persistência da Gold

Nesta etapa, as tabelas analíticas finais são persistidas na camada Gold do Amazon S3.

As saídas são armazenadas em formato Parquet, mantendo o mesmo padrão de otimização adotado na Silver e permitindo consumo eficiente por ferramentas analíticas, dashboards e aplicações futuras.

In [0]:
# Valida o caminho de destino antes da persistência para evitar sobrescrita acidental das camadas Bronze ou Silver.

assert GOLD_PATH != SILVER_PATH, \
    "ERRO: Gold e Silver apontam para o mesmo caminho."

assert "/gold/" in GOLD_PATH, \
    "ERRO: destino não corresponde à camada Gold."

print("[OK] Caminho de escrita da Gold validado.")

In [0]:
# Persiste as tabelas analíticas finais da Gold em formato Parquet no Amazon S3.

for nome, df in TABELAS_GOLD.items():
    caminho_destino = f"{GOLD_PATH}{nome}/"

    (
        df.write
        .mode("overwrite")
        .parquet(caminho_destino)
    )

    print(f"[OK] {nome} gravada em {caminho_destino}")

In [0]:
# Relê as tabelas persistidas em Parquet para validar a disponibilidade e integridade das saídas da camada Gold.

tabelas_gold_persistidas = {}

for nome in TABELAS_GOLD.keys():
    caminho = f"{GOLD_PATH}{nome}/"

    df = spark.read.parquet(caminho)

    tabelas_gold_persistidas[nome] = df

    print(
        f"[OK] {nome}: "
        f"{df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

## 9. Validação final e metadados da Gold

Nesta etapa são validadas as tabelas Gold após a persistência no Amazon S3.

São verificadas a quantidade de registros, a unicidade das chaves e a consistência entre as tabelas geradas em memória e os arquivos Parquet persistidos. Também são registrados metadados da execução para monitoramento e evidências.

In [0]:
# Compara as tabelas Gold geradas em memória com os arquivos persistidos para garantir que não houve perda de registros durante a escrita.

comparacao_gold = []

for nome, df_original in TABELAS_GOLD.items():
    df_persistido = tabelas_gold_persistidas[nome]

    registros_gerados = df_original.count()
    registros_persistidos = df_persistido.count()

    comparacao_gold.append({
        "tabela": nome,
        "registros_gerados": registros_gerados,
        "registros_persistidos": registros_persistidos,
        "diferenca_registros": registros_persistidos - registros_gerados
    })

df_comparacao_gold = spark.createDataFrame(comparacao_gold)

display(df_comparacao_gold.orderBy("tabela"))

In [0]:
# Valida a unicidade das chaves nas tabelas Gold persistidas em Parquet.

resultado_chaves_persistidas = []

for nome, chaves in CHAVES_GOLD.items():
    df = tabelas_gold_persistidas[nome]

    duplicadas = (
        df
        .groupBy(*chaves)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    resultado_chaves_persistidas.append({
        "tabela": nome,
        "chaves_duplicadas": duplicadas
    })

df_chaves_gold_persistidas = spark.createDataFrame(
    resultado_chaves_persistidas
)

display(df_chaves_gold_persistidas.orderBy("tabela"))

In [0]:
# Registra os metadados da execução da camada Gold para monitoramento.

from datetime import datetime, timezone

data_hora_processamento = datetime.now(timezone.utc)

metadados_gold = []

for nome, df in tabelas_gold_persistidas.items():
    metadados_gold.append({
        "tabela": nome,
        "registros": df.count(),
        "colunas": len(df.columns),
        "camada": "gold",
        "formato": "parquet",
        "status": "SUCESSO",
        "data_hora_processamento_utc": data_hora_processamento
    })

df_metadados_gold = spark.createDataFrame(metadados_gold)

display(df_metadados_gold.orderBy("tabela"))

## 10. Aplicações da Gold em Inteligência Artificial

As tabelas da camada Gold foram estruturadas para permitir consumo direto por aplicações analíticas e futuras soluções de Inteligência Artificial.

Entre as possíveis aplicações estão:

### Predição de risco educacional

Os indicadores históricos de alfabetização, participação, proficiência e evolução podem ser utilizados como variáveis de entrada para modelos capazes de identificar municípios com maior risco de apresentar resultados abaixo das metas futuras.

Exemplos de variáveis:
- taxa de alfabetização;
- evolução anual;
- percentual de participação;
- proficiência média;
- percentual de alunos alfabetizados;
- distância para a meta.

### Classificação de municípios prioritários

Modelos de classificação ou técnicas de agrupamento podem utilizar os indicadores da Gold para identificar municípios com características semelhantes e criar grupos de prioridade para intervenção educacional.

Exemplos:
- municípios com queda de desempenho;
- municípios persistentemente abaixo da meta;
- localidades com baixa participação nas avaliações;
- localidades com baixa proficiência média.

### Previsão da taxa de alfabetização

Com a inclusão de novos anos na pipeline, os dados históricos podem alimentar modelos de regressão ou séries temporais para estimar taxas futuras de alfabetização e avaliar a probabilidade de cumprimento das metas estabelecidas.

### Sistemas de apoio à decisão

As tabelas Gold também podem alimentar sistemas analíticos ou assistentes baseados em IA capazes de responder perguntas sobre o desempenho educacional, como:

- quais municípios estão mais distantes das metas;
- quais apresentaram maior evolução;
- onde houve queda no desempenho;
- quais localidades devem receber maior atenção.

### Evolução futura

Para uso produtivo em modelos de Machine Learning, seria necessário ampliar o histórico temporal e incorporar outras variáveis explicativas, como características socioeconômicas, infraestrutura escolar, formação docente e investimento público.

A arquitetura implementada permite essa evolução sem alterar as camadas de dados brutos, mantendo a Gold como fonte de features analíticas para aplicações futuras.

## 11. Aplicações da Gold em Políticas Públicas

Os indicadores disponibilizados na camada Gold podem apoiar o planejamento, a priorização e o acompanhamento de políticas públicas voltadas à alfabetização.

### Identificação de municípios prioritários

A comparação entre a taxa de alfabetização observada e as metas permite identificar municípios com maior distância em relação aos resultados esperados.

Indicadores como `diferenca_para_meta`, `status_meta` e `evolucao_taxa` podem apoiar a priorização de localidades que apresentam baixo desempenho ou queda nos resultados.

### Acompanhamento das metas de alfabetização

A visão nacional e municipal permite acompanhar o cumprimento das metas ao longo do tempo, identificando:

- municípios que atingiram ou superaram a meta;
- municípios que permanecem abaixo da meta;
- distância entre o resultado observado e o objetivo estabelecido;
- evolução dos indicadores entre diferentes anos.

### Análise da participação nas avaliações

Os indicadores derivados da base de alunos permitem acompanhar presença e participação válida nas avaliações.

Baixos percentuais de participação podem indicar a necessidade de investigar fatores que dificultam a realização das avaliações e também devem ser considerados na interpretação dos indicadores de desempenho.

### Identificação de desigualdades educacionais

As visões por município, UF e rede de ensino permitem comparar diferentes contextos educacionais e identificar localidades ou redes com resultados significativamente inferiores.

Essas informações podem apoiar análises para direcionamento de programas, recursos e ações educacionais.

### Avaliação da evolução das políticas

Com a incorporação de novos anos à pipeline, será possível acompanhar se municípios inicialmente classificados como prioritários apresentam melhora após ações ou programas educacionais.

A evolução histórica dos indicadores pode servir como apoio à avaliação de resultados das políticas implementadas.

### Apoio à tomada de decisão

A camada Gold fornece dados analíticos que podem ser consumidos por dashboards e sistemas de apoio à decisão, permitindo que gestores acompanhem indicadores de alfabetização e identifiquem localidades que demandam maior atenção.

Os indicadores produzidos não determinam automaticamente a alocação de recursos ou a adoção de políticas. Eles funcionam como evidências quantitativas que devem ser analisadas em conjunto com informações socioeconômicas, territoriais e educacionais adicionais.